# 파일 기반 질의 & 관리

`upload_file()`, `get_file()`, `list_files()`, `delete_file()` — Files API 관리.
`Part.from_uri()`, `Part.from_bytes()`, `Part.from_file()` — 파일 참조/인라인 전달.

In [1]:
from llm_manager._internal import handler_client as llm_handler
from llm_manager._internal.handler_client import call

# llm_handler.configure(base_url="http://localhost:8100")
from llm_manager._internal.handler_client import upload_file, get_file, list_files, delete_file, Part


In [2]:
from llm_manager._internal.handler_client import upload_file, get_file, list_files, delete_file

# 테스트할 파일 경로로 변경
FILE_PATH = "/home/junyoung_oh/projects/kiwoom/kiwoom-rag/docs/키움투자자산운용-최종 데모발표.pdf"

# 1. Files API로 업로드 → FileRef 반환
file_ref = upload_file(FILE_PATH, display_name="테스트문서")
print(f"name: {file_ref.name}")
print(f"uri: {file_ref.uri}")
print(f"state: {file_ref.state}")
print(f"display_name: {file_ref.display_name}")
print(f"size_bytes: {file_ref.size_bytes}")
print(f"expiration_time: {file_ref.expiration_time}")

# 2. contents 리스트로 텍스트 + 파일 전달
response = call(
    ["이 문서의 핵심 내용을 3줄로 요약해줘", file_ref],
)
print(response.text)


name: files/lrjvv1vecu3b
uri: https://generativelanguage.googleapis.com/v1beta/files/lrjvv1vecu3b
state: ACTIVE
display_name: upload
size_bytes: 13914476
expiration_time: 2026-04-18 06:47:35.469614+00:00
이 문서의 핵심 내용을 3줄로 요약해 드립니다.

1. **AI 내재화 목표:** 키움투자자산운용은 업무 생산성 증대와 휴먼에러 방지를 위해 리스크 관리, 글로벌 마켓, ESG 분야에 특화된 'AI 어시스턴트 및 에이전트'를 도입합니다.
2. **업무 효율성 증대:** 수작업으로 진행되던 데이터 취합 및 보고서 작성을 자동화하여 리스크 관리(24→8시간), ESG 평가(214→21시간) 등 주요 업무 시간을 획기적으로 단축했습니다.
3. **향후 로드맵:** 2026년 상반기 내 주요 에이전트를 순차 도입하고, 2027년까지 멀티 에이전트 시스템 및 AI 네이티브 플랫폼 구축을 통해 업무 프로세스를 고도화할 계획입니다.


### 파일 관리 — `get_file()`, `list_files()`, `delete_file()`

업로드된 파일 조회, 목록 확인, 삭제:

In [6]:
# 파일 존재 확인 (없으면 None 반환)
info = get_file(file_ref.name)
if info:
    print(f"파일 존재: {info.display_name} ({info.size_bytes} bytes)")
else:
    print("파일이 만료됨 — 재업로드 필요")

# 전체 파일 목록
files = list_files()
print(f"\n전체 파일 수: {len(files)}개")
for f in files:
    print(f"  {f.display_name}: {f.name}")

# 파일 삭제 (이미 삭제된 경우에도 에러 없음)
delete_file(file_ref.name)
print("\n파일 삭제 완료")


파일이 만료됨 — 재업로드 필요


HandlerError: {"detail":"Method Not Allowed"} (405)

### 공개 URL 직접 참조 — `Part.from_uri()` - mime_type = "application/pdf" 같은 mime_type 지정 필수 

업로드 없이 공개 URL을 직접 전달합니다. Google 서버가 해당 URL에서 파일을 가져옵니다.

In [5]:
from llm_manager._internal.handler_client import Part

# 공개 PDF URL을 직접 전달 (Google 서버가 URL에서 파일을 가져옴)
response = call(
    [
        "이 논문의 핵심 기여를 한 문장으로 요약해줘",
        Part.from_uri("https://arxiv.org/pdf/1706.03762", "application/pdf"),
    ],
)
print(response.text)

이 논문의 핵심 기여는 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN)을 완전히 배제하고, 오직 **어텐션(Attention) 메커니즘만을 기반으로 하여** 병렬 처리가 가능하고 학습 효율성과 성능이 뛰어난 새로운 신경망 아키텍처인 **트랜스포머(Transformer)를 제안**한 것입니다.


### 로컬 파일 바이너리 전달 — `Part.from_bytes()`

로컬 파일을 읽어서 bytes로 직접 전달합니다. 업로드 없이 요청 본문에 포함됩니다.

> 소규모 파일(이미지, 짧은 PDF 등)에 적합합니다. 대용량 파일은 `upload_file()`을 사용하세요.

In [ ]:
# 테스트할 로컬 파일 경로로 변경
LOCAL_FILE = "/home/junyoung_oh/projects/kiwoom/kiwoom-rag/packages/llm-handler/quickstarts/photo1.png"

# 로컬 파일을 bytes로 읽어서 전달 (업로드 불필요)
with open(LOCAL_FILE, "rb") as f:
    image_bytes = f.read()

response = call(
    ["이 이미지에 무엇이 있는지 설명해줘", Part.from_bytes(image_bytes)],
)
print(response.text)

이 이미지는 온라인 체스 플랫폼(Chess.com 스타일)에서 진행 중인 **체스 퍼즐(묘수풀이)** 화면입니다. 구체적인 내용은 다음과 같습니다:

### **1. 기물 배치 상황**
*   **백색(White):** 킹(e1), 룩(f2), 비숍 2개(c3, h5)가 있습니다.
*   **흑색(Black):** 킹(e4), 퀸(d7)이 있습니다.

### **2. 게임 정보 및 UI**
*   **상단 제목:** "Study - Win"이라고 되어 있어, 백색이 이기는 방법을 찾는 '엔드게임 스터디'임을 알 수 있습니다.
*   **플레이어 이름:** 하단 플레이어 이름은 "Rinck"으로 표시되어 있습니다. 이는 유명한 체스 묘수풀이 작가인 앙리 린크(Henri Rinck)의 작품임을 시사합니다.
*   **평가 막대:** 왼쪽의 수직 바와 하단의 **'7.7'**이라는 숫자는 컴퓨터 엔진 분석 결과 백색이 매우 승산이 높음(압도적으로 유리함)을 나타냅니다.
*   **기물 이득:** 하단에 '+2'라고 표시되어 있어, 기물 점수상으로도 백색이 약간 앞서고 있습니다.

### **3. 상황 분석 (체스적 관점)**
이 국면은 백색이 자신의 기물들을 이용해 흑색의 퀸을 잡아내거나 체크메이트를 노리는 퍼즐입니다. 
*   **핵심 수:** 이 퍼즐의 정답은 **Be8!**입니다. 
    *   만약 흑색이 퀸으로 비숍을 잡으면(**Qxe8**), 백색은 **Re2+**로 체크를 걸어 킹과 퀸을 동시에 공격하는 '스큐어(Skewer)' 전술로 퀸을 잡아내고 승리하게 됩니다.

전반적으로 백색의 정교한 기물 협동을 통해 강력한 흑색 퀸을 무력화시키는 전술 연습 장면입니다.


### 여러 파일 동시 전달

`contents` 리스트에 여러 파일과 텍스트를 자유롭게 조합할 수 있습니다.

In [5]:
# 여러 파일 + 텍스트를 하나의 contents 리스트로
response = call(
    [
        "이 두 논문의 핵심 차이를 비교해줘",
        Part.from_uri("https://arxiv.org/pdf/1706.03762", "application/pdf"),
        Part.from_uri("https://arxiv.org/pdf/1810.04805", "application/pdf"),
    ],
)
print(response.text)

제시된 두 논문 **"Attention Is All You Need" (Transformer)**와 **"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" (BERT)**는 현대 자연어 처리(NLP)의 근간을 이루는 매우 중요한 연구들입니다.

BERT가 Transformer의 구조를 기반으로 만들어졌기 때문에 밀접한 관련이 있지만, 목적과 구조적 활용 방식에서 다음과 같은 핵심적인 차이가 있습니다.

---

### 1. 주요 목표 및 공헌 (Main Goal)
*   **Transformer:** **새로운 신경망 아키텍처의 제안**입니다. 기존의 RNN이나 CNN을 완전히 배제하고 오직 'Attention' 메커니즘만 사용하여 시퀀스 데이터를 처리할 수 있음을 증명했습니다. 주로 기계 번역 성능 향상과 연산 속도 개선에 집중했습니다.
*   **BERT:** **범용적인 언어 표현(Representation)을 학습하기 위한 사전 학습(Pre-training) 방법론**의 제안입니다. Transformer의 구조를 활용하여 방대한 양의 텍스트로부터 언어의 맥락을 미리 학습하고, 이를 다양한 하위 작업(NLP tasks)에 쉽게 적용(Fine-tuning)하는 데 초점을 맞췄습니다.

### 2. 모델 구조 (Architecture Structure)
*   **Transformer:** **인코더(Encoder)와 디코더(Decoder)**가 결합된 구조입니다. 입력 문장을 이해하는 부분(인코더)과 이를 바탕으로 새로운 문장(예: 번역문)을 생성하는 부분(디코더)이 모두 포함되어 있습니다.
*   **BERT:** Transformer의 **인코더(Encoder) 스택만을 사용**합니다. 문장을 생성하는 기능보다는 주어진 텍스트의 의미를 깊게 이해하고 수치화(임베딩)하는 데 특화되어 있습니다.

### 3. 문맥 방향성 (Directionality)
*   *

### 미디어 해상도 제어 — `media_resolution=`

이미지/PDF의 처리 해상도를 설정합니다. 높을수록 세밀한 분석이 가능하지만 토큰 사용량이 증가합니다.

| 값 | 이미지 토큰 | 동영상 토큰 | PDF 토큰 | 용도 |
| --- | --- | --- | --- | --- |
| `"low"` | 280 | 70 | 280 + 텍스트 | 빠른 분류, 대략적 분석 |
| `"medium"` | 560 | 70 | 560 + 텍스트 | 일반적 분석 |
| `"high"` | 1120 | 280 | 1120 + 텍스트 | 세밀한 텍스트 읽기, 상세 분석 |


### 기본값 (`media_resolution=None` 일 때)

SDK 가 `media_resolution` 인자를 명시하지 않으면 Gemini 가 `UNSPECIFIED` 로 처리:

| 미디어 | UNSPECIFIED 토큰 |
| --- | --- |
| 이미지 | 1120 |
| 동영상 | 70 |
| PDF | 560 |


In [ ]:
# 고해상도로 이미지 내 텍스트까지 세밀하게 분석
response = call(
    [
        "이 문서에서 표에 적힌 숫자들을 모두 추출해줘",
        Part.from_uri("https://arxiv.org/pdf/1706.03762", "application/pdf"),
    ],
    media_resolution="high",
)
print(response.text)

### 오디오 파일 질의 — `Part.from_file()`

로컬 오디오 파일을 직접 전달합니다. MIME 타입은 확장자 + 매직 바이트에서 자동 감지됩니다.

> Gemini 지원 오디오: `audio/wav`, `audio/mp3`, `audio/aiff`, `audio/aac`, `audio/ogg`, `audio/flac`  
> `.m4a`(컨테이너: MP4, 코덱: AAC)는 비공식이지만 동작합니다.

In [6]:
from llm_manager._internal.handler_client import Part

# 오디오 파일 질의 테스트
AUDIO_FILE = "./test_audio.m4a"

response = call(
    ["이 오디오의 내용을 요약해줘", Part.from_file(AUDIO_FILE)],
)
print(response.text)

제시된 오디오는 두 남성이 사업 및 자금 조달 상황에 대해 나누는 대화 내용을 담고 있습니다. 주요 내용을 요약하면 다음과 같습니다.

### 1. 코랑코(Koranko)와의 협상 상황
* 코랑코 측에서 당초 오늘 중으로 승인(OK) 여부를 알려주기로 했으나, 일정이 하루 밀려 **내일 확답**을 주기로 했습니다.
* 코랑코 측에서 '후단(추가 자금 조달)' 부분까지 전부 책임져 줄지가 핵심 관건입니다.

### 2. 자금 조달 및 계약금
* 현재 **약 60억 원 정도의 계약금**은 준비되어 있는 상태입니다.
* 하지만 나머지 잔금이 워낙 큰 액수라, 코랑코가 이 부분을 해결해주지 못할 경우 2개월 내에 자금을 모두 확보하는 데 어려움이 있을 것으로 예상하고 있습니다.

### 3. 키움증권 및 외부 압박
* 키움 측에서 계속 연락이 오고 있으며, 이미 법무법인까지 선임되어 조율 중인 상황이라 빠르게 회신을 줘야 하는 압박이 있습니다.
* 키움 팀장도 입장이 난처한 상황이라 내일 코랑코의 답변이 나오는 대로 미팅을 진행하거나 입장을 정리해야 합니다.

### 4. 대안 및 기타 사항
* 코랑코가 안 될 경우 FNI를 통한 방법도 고려해 볼 수 있으나, 이 경우 **약 200억 원 규모의 자금**이 필요하여 실질적으로 조달이 쉽지 않다는 점을 토로하고 있습니다.
* 어제 남동공단 현장에 관련 실무진(팀장, 본부장 등)이 직접 방문하여 상황을 확인했다는 언급도 포함되어 있습니다.

**결론적으로,** 현재 모든 계획이 **내일 나올 코랑코의 확답**에 달려 있으며, 그 결과에 따라 키움증권과의 관계 정리 및 후속 대책을 결정할 예정인 긴박한 상황으로 보입니다.
